# Point cloud processing

This section shows the practices employed to assess the point cloud received from the camera topic /camera/points. The inner machinations of the specific functions used are out of scope of this project, their influence will be highlighted here to show their importance.

To do this, a sample point cloud has been saved from the gazebo simulation and will be the subject of the filtering pipeline using the Open3D python library:
- steps:
    * Crop the point cloud to include only points below a given height.
    * Copy a downsampled version of the pointcloud to speed up future calculations.
    * Iteratively detect planes and remove corresponding points from both point clouds.
    From there, the downsampled point cloud will not be used anymore.
    * Detect clusters from the remaining points.
    * Create bounding boxes for these clusters.
    * Reject boxes that overlap with the costmap.

Any remaining bounding boxes are published as detected objects.

The original point cloud can be viewed using the widget below:

In [1]:
pcd_HQ = o3d.geometry.PointCloud()
pcd_HQ.points = o3d.utility.Vector3dVector(points_np)

pcd_LQ = pcd_HQ.voxel_down_sample(voxel_size=0.01)
object_pcd = o3d.geometry.PointCloud()
object_points = np.empty((0, 3))
points = np.asarray(pcd_HQ.points)
remaining_mask = np.ones(len(points), dtype=bool)

while True:
    if len(pcd_LQ.points) < 100:
        break

    plane_model, inliers = pcd_LQ.segment_plane(
        distance_threshold=0.003,
        ransac_n=3,
        num_iterations=2000
    )

    if len(inliers) < 50:
        break

    pcd_LQ = pcd_LQ.select_by_index(
        inliers,
        invert=True
    )

    dist = np.abs(
        plane_model[0]*points[:,0] +
        plane_model[1]*points[:,1] +
        plane_model[2]*points[:,2] +
        plane_model[3]
    ) / np.sqrt(
        plane_model[0]**2 +
        plane_model[1]**2 +
        plane_model[2]**2
    )

    plane_mask = dist < 0.004

    remaining_mask &= ~plane_mask
object_points = points[remaining_mask]
object_pcd.points = o3d.utility.Vector3dVector(object_points)

In [ ]:
from sklearn.cluster import DBSCAN

<model-viewer
    src="./models/output_3d_object.glb"
    camera-controls
    auto-rotate
    style="width: 640px; height: 640px; background: #d1d9e6;">
</model-viewer>
""")

In [ ]:
labels = clustering.labels_
unique_labels = set(labels)

for label in unique_labels:

    if label == -1:
        continue

    cluster_points = object_points[
        labels == label
    ]

    if len(cluster_points) < 3:
        continue

    cluster_pcd = o3d.geometry.PointCloud()

    cluster_pcd.points = (
        o3d.utility.Vector3dVector(
            cluster_points
        )
    )

    try:
        obb = (
            cluster_pcd
            .get_oriented_bounding_box()
        )

    except RuntimeError:

        continue

    center = obb.center
    extent = obb.extent

 